In [ ]:
%pip install -q duckdb numpy numba matplotlib google-cloud-storage google-auth

In [ ]:
import os
import sys

REPO_URL = "https://github.com/payamdav/pycrypto.git"
REPO_NAME = "pycrypto"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

REPO_PATH = os.path.abspath(REPO_NAME)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

# Add GCS tools to path
sys.path.insert(0, os.path.join(REPO_PATH, "packages/tools/google_cloud_storage_tools"))

import time
import io
import numpy as np
import matplotlib.pyplot as plt
from packages.candle_loader import load_candles
from packages.indicators import rolling_robust_z_score, ma, rsi_1_1
from gcs_tools import write_file, gcs_json_key_file

gcs_json_key_file()

In [ ]:
ASSET     = "btcusdt"
DATE_FROM = "2024-01-01 00:00:00"
DATE_TO   = "2026-01-01 00:00:00"

In [ ]:
candles = load_candles(ASSET, DATE_FROM, DATE_TO, columns=["c", "v", "q", "vwap", "vb", "vs"])
# columns: ts=0, c=1, v=2, q=3, vwap=4, vb=5, vs=6

In [ ]:
ZSCORE_WINDOW = 7 * 24 * 60
t0 = time.perf_counter()
v_zscore = rolling_robust_z_score(candles[:, 2], window=ZSCORE_WINDOW)
elapsed = time.perf_counter() - t0
print(f"rolling_robust_z_score elapsed: {elapsed:.4f}s  shape: {v_zscore.shape}")
candles = np.hstack([candles, v_zscore.reshape(-1, 1)])
# column 8 (index 7): rolling_robust_z_score of volume

In [ ]:
MA_WINDOW = 7 * 24 * 60
t0 = time.perf_counter()
v_ma = ma(candles[:, 2], window=MA_WINDOW)
elapsed = time.perf_counter() - t0
print(f"ma elapsed: {elapsed:.4f}s  shape: {v_ma.shape}")
candles = np.hstack([candles, v_ma.reshape(-1, 1)])
# column 9 (index 8): ma of volume

In [ ]:
t0 = time.perf_counter()
rsi_7 = rsi_1_1(candles[:, 4], window=7)
elapsed = time.perf_counter() - t0
print(f"rsi_1_1(window=7) elapsed: {elapsed:.4f}s  shape: {rsi_7.shape}")
candles = np.hstack([candles, rsi_7.reshape(-1, 1)])
# column 10 (index 9): rsi_1_1 of vwap window=7

In [ ]:
t0 = time.perf_counter()
rsi_14 = rsi_1_1(candles[:, 4], window=14)
elapsed = time.perf_counter() - t0
print(f"rsi_1_1(window=14) elapsed: {elapsed:.4f}s  shape: {rsi_14.shape}")
candles = np.hstack([candles, rsi_14.reshape(-1, 1)])
# column 11 (index 10): rsi_1_1 of vwap window=14

In [ ]:
t0 = time.perf_counter()
rsi_60 = rsi_1_1(candles[:, 4], window=60)
elapsed = time.perf_counter() - t0
print(f"rsi_1_1(window=60) elapsed: {elapsed:.4f}s  shape: {rsi_60.shape}")
candles = np.hstack([candles, rsi_60.reshape(-1, 1)])
# column 12 (index 11): rsi_1_1 of vwap window=60

In [ ]:
# Trimming
TRIM = 7 * 24 * 60
before = candles.shape
candles = candles[TRIM:]
print(f"shape before trimming: {before}  after trimming: {candles.shape}")

In [ ]:
GCS_BUCKET = "payamdprojectbucket"
GCS_KEY    = "lookback_lookahead_data"

buf = io.BytesIO()
np.save(buf, candles)
buf.seek(0)
file_size = buf.getbuffer().nbytes

t0 = time.perf_counter()
write_file(GCS_BUCKET, GCS_KEY, buf, content_type="application/octet-stream")
elapsed = time.perf_counter() - t0

print(f"Written to gs://{GCS_BUCKET}/{GCS_KEY}")
print(f"File size : {file_size / 1024 / 1024:.2f} MB  ({file_size:,} bytes)")
print(f"Write time: {elapsed:.4f}s")

## Final `candles` Array

Shape: `(N, 12)` — N rows × 12 columns (1-minute candles for BTCUSDT, 2024-01-01 → 2026-01-01, first TRIM=7×24×60 rows removed)

| Index | Column | Description |
|-------|--------|-------------|
| 0 | `ts` | Candle open time (UTC milliseconds) |
| 1 | `c` | Close price |
| 2 | `v` | Base asset volume |
| 3 | `q` | Quote asset volume |
| 4 | `vwap` | Volume-weighted average price |
| 5 | `vb` | Taker buy base volume (aggressive buys) |
| 6 | `vs` | Taker sell base volume (aggressive sells) |
| 7 | `v_zscore` | Rolling robust z-score of volume (window = 7×24×60) |
| 8 | `v_ma` | Moving average of volume (window = 7×24×60) |
| 9 | `rsi_7` | RSI scaled to [−1, 1] of vwap (window = 7) |
| 10 | `rsi_14` | RSI scaled to [−1, 1] of vwap (window = 14) |
| 11 | `rsi_60` | RSI scaled to [−1, 1] of vwap (window = 60) |

The first 7 days (TRIM rows) are removed to avoid zero-padded indicator values at the start.
The array is saved as a NumPy `.npy` file to GCS at `gs://payamdprojectbucket/lookback_lookahead_data`.